# 什么是智能体

## 基础概念

这样一个东西(源于对人的模仿), 感知环境变化, 并采取一定的行动
> 智能体被定义为任何能够通过传感器（Sensors）感知其所处环境（Environment），并自主地通过执行器（Actuators）采取行动（Action）以达成特定目标的实体。

其中的要点
- 环境
- 行动

即建立环境到行动的映射, 这个过程是**智能**的核心体现

## 发展路径

基线: 简单的反射智能体, 基于预定义的规则由传感器数据触发行动

- 基于模型的: 内部维持一个世界模型, 处理环境中无法被传感器直接感知的部分, 兼具记忆和预测功能. eg. 卡尔曼滤波器
- 基于目标的: 具备一定的自主性
- 基于效用的: 权衡利弊, 衡量每个方案的优缺点
- 学习型智能体: 基于 RL

## 新范式
llm-based agent

对比

传统智能体基于预定义的规则, 行为是有边界的且具有确定性, 泛化性弱. eg. 自动化工具

llm-based agent在丰富的数据上训练, 行为是概率性的, 泛化性强

## 衡量指标
- 内部决策机制
- 反应时间: 类似于人体的条件反射和非条件反射
- 知识的表达: 符号主义; 亚符号主义; 神经符号主义

> 其内核是一个巨大的神经网络，使其具备模式识别和语言生成能力。然而，当它工作时，它会生成一系列结构化的中间步骤，如思想、计划或 API 调用，这些都是明确的、可操作的符号。通过这种方式，它实现了感知与认知、直觉与理性的初步融合。

# 智能体的构成

## PEAS模型

- performance: 性能量度
- environment
- Actuators
- sensors

特别地, 当前阶段的 llm-based agent 面临的环境是**数字环境**, 即不是来自ee里面的一堆传感器, 而是通过api调用挖掘挖掘的数据
- 环境是部分可观测的, 无法像像 all-in-sensors 那样, 一下子处理所有的数据, 需要根据情况一点点的搜集.
> 要求agent具有记忆能力和探索(plan+skill)能力
- 上升到 MAS, 并行环境中要解决**并发问题**
- 处理时序问题, emmmm存在同步和互斥关系

## agent-loop

![](https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/1-figures/1757242319667-5.png)

从宏观视角看, agent与env的交互主要通过obervation, action. 不断循环进行

如图, 是从agent的视角来看. 
- perception
- thought: 现阶段, 分为plan how to do / plan what to do
- action

# interaction protocol

llm输出的是文本, 在循环中, 如何交接每round中生成的数据

# 实践

In [1]:
%cd ../../ 
import os
os.getcwd()

/Users/yqz/project/hello-agents


'/Users/yqz/project/hello-agents'

In [2]:
from dotenv import load_dotenv
load_dotenv()
os.getenv("TAVILY_API_KEY")

'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'

In [75]:
SYSTEM_PROMPT = """
You are an intelligent travel assistant. You must analyze the user request and solve it step by step using the available tools.

Available tools:
- get_weather(city: str): Query the real-time weather of the specified city.
- get_attraction(city: str, weather: str): Recommend attractions based on the city and the weather.

OUTPUT FORMAT (STRICT):
You MUST reply with EXACTLY ONE valid JSON object and NOTHING ELSE.
No Markdown. No code fences. No explanations. No extra text.

The JSON protocol is:

Top-level keys (ONLY):
- "thought": string
- "todo": object

"todo" object:
- "type": "tool" OR "output"
- "name": string
- "args": object OR string (depending on type)

Rules:
1) If todo.type == "tool":
   - todo.name MUST be exactly one of: "get_weather", "get_attraction"
   - todo.args MUST be a JSON object containing the keyword arguments for that tool
   Examples:
   {"thought":"...","todo":{"type":"tool","name":"get_weather","args":{"city":"Beijing"}}}
   {"thought":"...","todo":{"type":"tool","name":"get_attraction","args":{"city":"Beijing","weather":"Sunny"}}}

2) If todo.type == "output":
   - todo.name MUST be exactly "finish"
   - todo.args MUST be a string containing the final answer for the user
   Example:
   {"thought":"...","todo":{"type":"output","name":"finish","args":"...final answer..."}} 

Strict JSON requirements:
- Double quotes for all keys and strings
- No trailing commas
- No comments
- No extra top-level keys besides "thought" and "todo"

Decision policy:
- If you need tool information, output a tool action.
- If you have enough information, output an output action and finish.

"""


## 搭建工具

In [4]:
import requests

def get_weather(city: str) -> str:
    """
    通过调用 wttr.in API 查询真实的天气信息。
    """
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        data = response.json()
        
        # 提取当前天气状况
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        
        # 格式化成自然语言返回
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
        
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"


In [5]:
get_weather(city = "Beijing")

'Beijing当前天气:Sunny，气温5摄氏度'

In [6]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"


In [7]:
get_attraction(
    city = "Beijing", 
    weather = get_weather("Beijing")
)

'在北京阳光明媚、气温5摄氏度的天气下，故宫和颐和园是最值得去的旅游景点。这些地方不仅历史悠久，还适合户外活动。'

In [8]:
# 将所有工具函数放入一个字典，方便后续调用
# 字典中, 键为函数名, 值为函数
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

## 接入LLM

In [9]:
from openai import OpenAI
import os

class OpenaiClient:
    def __init__(self, model:str, api_key:str, base_utl:str):
        self.model = model
        self.client = OpenAI(
            api_key=api_key,
            base_url=base_utl
        )
    

    def generate(self, prompt:str, system_prompt:str):
        message = [
            {
                "role": "system", 
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        reponse = self.client.chat.completions.create(
            model = self.model,
            messages= message
        )

        return reponse.choices[0].message.content

In [10]:
myllm = OpenaiClient(
    api_key=os.getenv("OLLAMA_API_KEY"),
    base_utl=os.getenv("OLLAMA_BASE_URL"),
    model = "qwen3:0.6b"
)

In [11]:
os.getenv("OLLAMA_BASE_URL")

'http://127.0.0.1:11434/v1'

In [51]:
output = myllm.generate(
    prompt="I want to go to Beijing, please recommend some scenic spots for me",
    system_prompt=SYSTEM_PROMPT
)

In [54]:
json.loads(output.strip())['todo']['type']

'tool'

## 开始测试

### 协议制定

#### 上下文管理

llm输出格式定义

```txt
{
  "thought": "I need to check the weather in Beijing first.",
  "todo": {
    "type": "tool",
    "name": "get_weather",
    "args": { "city": "Beijing" }
  }
}
```

```txt
{
  "thought": "I have finished",
  "todo": {
     "type": "output",
     "name": "finish",
     "args": "recommend you ......"
  }
}
```

In [70]:
history_prompt = []
prompt="I want to go to Beijing, please recommend some scenic spots for me",

In [71]:
history_prompt.append(
    json.dumps({
        "role": "user",
        "content": prompt
    }, ensure_ascii = False)
)

In [72]:
myllm = OpenaiClient(
    api_key=os.getenv("OLLAMA_API_KEY"),
    base_utl=os.getenv("OLLAMA_BASE_URL"),
    model = "qwen3:0.6b"
)

In [ ]:
for i in range(10):
    # 第 i 轮分割线
    print('=' * 20 + f"第{i+1}轮" + '=' * 20)
    # 整理提示词
    full_prompt = '\n'.join(history_prompt)
    print(f"input: {full_prompt}")

    # 输入给llm
    response = myllm.generate(
        prompt=full_prompt,
        system_prompt=SYSTEM_PROMPT
    )
    print(f"output: {response}")

    # 解析并输出
    response_json = json.loads(response)
    if response_json['todo']['type'] == 'output':
        print(f"final_answer: {response_json['todo'][args]}")
        break
    elif response_json['todo']['type'] == 'tool':
        tool_name = response_json['todo']['name']
        tool_args = response_json['todo']['args']
        result = available_tools[tool_name](**tool_args)
        print(f"tool: {result}")
        history_prompt.append(
            json.dumps(
                {
                    'role': 'tool',
                    'content': result
                }, ensure_ascii=False
            )
        )

====================第0轮====================
input: {"role": "user", "content": ["I want to go to Beijing, please recommend some scenic spots for me"]}
output: {"thought": "I need to use the get_attraction tool with city 'Beijing' to recommend scenic spots. However, I don't have the specific weather data to use it. I should first check the current weather in Beijing to use get_attraction properly.", "todo": {"type": "tool", "name": "get_attraction", "args": {"city": "Beijing", "weather": ""}}}
tool: 北京雨后的故宫和后海是最值得游览的景点，它们在不同天气下展现独特美景。秋季是北京最佳旅游时间，气候宜人，适合拍摄和户外活动。
====================第1轮====================
input: {"role": "user", "content": ["I want to go to Beijing, please recommend some scenic spots for me"]}
{"role": "tool", "content": "北京雨后的故宫和后海是最值得游览的景点，它们在不同天气下展现独特美景。秋季是北京最佳旅游时间，气候宜人，适合拍摄和户外活动。"}
output: {
  "thought": "The user wants to recommend scenic spots based on Beijing's weather information. I will use the get_attraction tool with the provided city and weather details to fi

NameError: name 'args' is not defined